# 1. Introduction

This project focuses on building a simple and practical deep learning model to predict customer churn in a telecom company.

Customer churn refers to when a customer stops using a service. Identifying such customers early helps businesses take action to retain them.

In this notebook, we use an Artificial Neural Network (ANN) to analyze customer data such as tenure, monthly charges, and service usage to predict whether a customer will leave or stay.

The workflow is designed to be clean, structured, and easy to explain.

# 2. Import Libraries

We import only the required libraries for data processing, visualization, and deep learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

sns.set_style('whitegrid')

# 3. Load Dataset

We load the dataset and inspect its basic structure.

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)
df.head()

# 4. Exploratory Data Analysis

We explore basic information like data types and churn distribution.

In [ ]:
print(df.dtypes)
print(df['Churn'].value_counts())

sns.countplot(x='Churn', data=df)
plt.show()

# 5. Data Preprocessing

We clean the dataset and prepare it for modeling.

In [ ]:
data = df.copy()

data['TotalCharges'] = data['TotalCharges'].replace(' ', np.nan)
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data['TotalCharges'] = data['TotalCharges'].fillna(data['TotalCharges'].median())

data['Churn'] = data['Churn'].map({'Yes':1, 'No':0})

data.drop('customerID', axis=1, inplace=True)

# 6. Feature Engineering

We create simple features to improve model understanding.

In [ ]:
data['charges_per_tenure'] = data['MonthlyCharges']/(data['tenure']+1)

X = data.drop('Churn', axis=1)
y = data['Churn']

X = pd.get_dummies(X, drop_first=True)

# 7. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 8. Build ANN Model

In [ ]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 9. Train Model

In [ ]:
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2)

# 10. Model Evaluation

In [ ]:
y_pred = (model.predict(X_test)>0.5).astype(int)

print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# 11. Visualization

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.show()

# 12. Prediction Function

In [ ]:
def predict(prob):
    if prob < 0.35:
        return 'Low Risk'
    elif prob < 0.65:
        return 'Medium Risk'
    else:
        return 'High Risk'

p = model.predict(X_test[0].reshape(1,-1))[0][0]
print(p, predict(p))

# 13. Conclusion

This project demonstrates how a simple ANN can be used to predict customer churn.

The model provides both predictions and risk levels, making it useful and easy to understand.